In [ ]:
!pip uninstall -y peft
!pip install evaluate setfit "transformers==4.39.3" "huggingface_hub<0.26.0"
!pip install --upgrade transformers peft accelerate

Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
  Using cached transformers-4.39.3-py3-none-any.whl.metadata (134 kB)
  Using cached huggingface_hub-0.25.2-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.39.3-py3-none-any.whl (8.8 MB)
Using cached huggingface_hub-0.25.2-py3-none-any.whl (436 kB)
Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing install

In [8]:
from datasets import load_dataset

# 准备数据并进行数据划分
tomatoes = load_dataset("rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]


In [9]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
import numpy as np
import evaluate
import torch

# 确保数据已加载
try:
    train_data, test_data = tomatoes["train"], tomatoes["test"]
except NameError:
    tomatoes = load_dataset("rotten_tomatoes")
    train_data, test_data = tomatoes["train"], tomatoes["test"]

# 加载模型和分词器
model_id = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

from transformers import TrainingArguments, Trainer

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [10]:
from transformers import TrainingArguments, Trainer

# 1. 定义训练参数
training_args = TrainingArguments(
    "model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to="none"
)

# 2. 初始化 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 3. 【核心修复】动态剔除名字里包含 "Notebook" 的回调组件，绕过进度条 Bug
# 注意：这里我们不需要 import 任何外部 callback 类了！
trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if "Notebook" not in cb.__class__.__name__
]

# 4. 现在可以绝对安全地运行初始评估了
print("Starting initial evaluation...")
initial_results = trainer.evaluate()
print("Initial evaluation results:", initial_results)

Starting initial evaluation...
Initial evaluation results: {'eval_loss': 0.760871410369873, 'eval_model_preparation_time': 0.036, 'eval_f1': 0.0, 'eval_runtime': 4.2235, 'eval_samples_per_second': 252.397, 'eval_steps_per_second': 15.864, 'epoch': 0}


In [11]:
# 加载模型和分词器
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 打印层的名称
for name, param in model.named_parameters():
    print(name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

In [12]:
# 冻结除分类头之外的所有结构
for name, param in model.named_parameters():
    # 可训练的分类头
    if name.startswith("classifier"):
        param.requires_grad = True
    # 冻结其他所有结构
    else:
        param.requires_grad = False

from transformers import TrainingArguments, Trainer

# 执行训练过程的Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,  # <--- 注意这里：将 tokenizer 修改为 processing_class
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


Step,Training Loss
500,0.685509


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=534, training_loss=0.6852788032217418, metrics={'train_runtime': 28.5579, 'train_samples_per_second': 298.692, 'train_steps_per_second': 18.699, 'total_flos': 227605451772240.0, 'train_loss': 0.6852788032217418, 'epoch': 1.0})

In [14]:
# 1. 补全缺失的导入
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer

# 载入模型和分词器
model_id = "bert-base-cased"
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. 冻结部分层的参数 (Layer Freezing)
# 编码器块11从索引165开始，我们冻结该块之前的所有结构
for index, (name, param) in enumerate(model.named_parameters()):
    if index < 165:
        param.requires_grad = False

# 3. 初始化 Trainer
trainer = Trainer(
    model=model,
    args=training_args,  # 确保你之前已经定义了 training_args
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ---------------------------------------------------------
# 【核心防崩溃补丁】每次新建 Trainer 后，必须立刻剔除带有 Bug 的 UI 组件！
trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if "Notebook" not in cb.__class__.__name__
]
# ---------------------------------------------------------

# 4. 执行训练并评估
print("🚀 开始微调模型...")
trainer.train()

print("\n📊 开始最终评估...")
final_results = trainer.evaluate()
print("🎉 最终评估结果:", final_results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 开始微调模型...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


📊 开始最终评估...
🎉 最终评估结果: {'eval_loss': 0.4100649952888489, 'eval_f1': 0.8072519083969466, 'eval_runtime': 3.3743, 'eval_samples_per_second': 315.922, 'eval_steps_per_second': 19.856, 'epoch': 1.0}


In [18]:
!pip install huggingface_hub==0.23.5

In [2]:
!pip install --upgrade transformers huggingface_hub setfit

  Using cached huggingface_hub-1.5.0-py3-none-any.whl.metadata (13 kB)
  Using cached setfit-1.1.3-py3-none-any.whl.metadata (12 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.23.5
    Uninstalling huggingface-hub-0.23.5:
      Successfully uninstalled huggingface-hub-0.23.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.3.0
    Uninstalling transformers-5.3.0:
      Successfully uninstalled transformers-5.3.0
  Attempting uninstall: setfit
    Found existing installation: setfit 1.0.3
    Uninstalling setfit-1.0.3:
      Successfully uninstalled setfit-1.0.3


In [1]:
from datasets import load_dataset, Dataset
import pandas as pd
from setfit import SetFitModel, Trainer, TrainingArguments

def sample_dataset(dataset, num_samples=16):
    df = dataset.to_pandas()
    # 完美保留 label 列，并安全重置索引
    sampled_df = df.groupby("label").sample(n=num_samples, random_state=42).reset_index(drop=True)
    return Dataset.from_pandas(sampled_df)

# 1. 检查并加载数据
try:
    train_ref = tomatoes["train"]
    test_ref = test_data
except NameError:
    tomatoes = load_dataset("rotten_tomatoes")
    train_ref = tomatoes["train"]
    test_ref = tomatoes["test"]

# 抽取小样本数据集
sample_train_data = sample_dataset(train_ref, num_samples=16)

# 2. 加载预训练的 SentenceTransformer 模型
model = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2")

# 3. 定义训练参数 (使用新版 TrainingArguments)
args = TrainingArguments(
    batch_size=16,
    num_epochs=3,
    eval_strategy="epoch", # 新版 transformers 推荐使用 eval_strategy
)

# 4. 创建训练器并训练 (使用新版 Trainer)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=sample_train_data,
    eval_dataset=test_ref,
    metric="f1"
)

print("🚀 开始小样本微调...")
trainer.train()

# 5. 在测试数据上评估模型
print("\n📊 开始评估...")
metrics = trainer.evaluate()
print(f"🎉 评估结果: {metrics}")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:268: DeprecationWarning: Read the `app_url` setting from the appropriate Settings object.
  app_url = wandb.util.app_url(tags["base_url"])  # type: ignore[index]
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:268: DeprecationWarning: Read the `app_url` setting from the appropriate Settings object.
  app_url = wandb.util.app_url(tags["base_url"])  # type: ignore[index]


/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)


TypeError: SentenceTransformerTrainer.compute_loss() got an unexpected keyword argument 'num_items_in_batch'

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM
from transformers import DataCollatorForLanguageModeling
from transformers import TrainingArguments, Trainer

# ---------------------- 0. 重新加载数据集 ----------------------
# 补上缺失的数据集加载步骤
tomatoes = load_dataset("rotten_tomatoes")
train_data = tomatoes["train"]
test_data = tomatoes["test"]

# ---------------------- 1. 加载预训练模型与分词器 ----------------------
model = AutoModelForMaskedLM.from_pretrained("bert-base-cased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# ---------------------- 2. 数据预处理函数 ----------------------
def preprocess_function(examples):
    # 对文本进行分词，截断超长文本
    return tokenizer(examples["text"], truncation=True)

# 对训练集和测试集进行分词处理，并移除标签（自监督任务无需情感标签）
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_train = tokenized_train.remove_columns("label")

tokenized_test = test_data.map(preprocess_function, batched=True)
tokenized_test = tokenized_test.remove_columns("label")

# ---------------------- 3. 定义掩码数据整理器 ----------------------
# 使用词元掩码（mlm_probability=0.15，随机掩码15%的词元）
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# ---------------------- 4. 配置训练参数 ----------------------
training_args = TrainingArguments(
    output_dir="nlm",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy="epoch",

    # 👇 新增下面这两行，开启文本进度播报
    logging_strategy="steps",  # 按步数记录日志
    logging_steps=50,          # 每跑 50 步（大约几十秒）就在屏幕上打印一次当前的 Loss 和进度

    report_to="none"
)

# ---------------------- 5. 初始化Trainer并训练 ----------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer, # 使用 processing_class 替代 tokenizer 避免警告
    data_collator=data_collator
)

# 【防崩溃补丁】移除会导致报错的 Notebook 进度条组件
trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if "Notebook" not in cb.__class__.__name__
]

print("🚀 开始掩码语言模型 (MLM) 训练...")

# 保存预训练分词器
tokenizer.save_pretrained("nlm")

# 开始训练
trainer.train()

# 保存微调后的模型
model.save_pretrained("nlm")
print("🎉 训练完成并已保存至 'nlm' 目录！")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

🚀 开始掩码语言模型 (MLM) 训练...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 训练完成并已保存至 'nlm' 目录！


In [5]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

# 1. 加载微调后的模型与分词器（路径为训练时的output_dir）
model_path = "nlm"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForMaskedLM.from_pretrained(model_path)
model.eval()  # 开启评估模式，关闭dropout

# 2. 构建推理样本（书中示例：What a [MASK] day!）
texts = [
    "What a [MASK] day!",
    "What a [MASK] film!",
    "What a [MASK] story!"
]

# 3. 批量推理函数
def predict_mask_token(text, tokenizer, model):
    # 分词并获取掩码位置
    inputs = tokenizer(text, return_tensors="pt")
    mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

    # 前向传播（无梯度计算，加速）
    with torch.no_grad():
        outputs = model(** inputs)
    logits = outputs.logits

    # 获取掩码位置的预测结果（取概率最高的词）
    mask_token_logits = logits[0, mask_token_index, :]
    predicted_token_id = mask_token_logits.argmax(dim=-1)
    predicted_token = tokenizer.decode(predicted_token_id)

    # 替换掩码并返回结果
    return text.replace("[MASK]", predicted_token)

# 4. 执行推理并打印结果
for text in texts:
    result = predict_mask_token(text, tokenizer, model)
    print(f"输入: {text:20} 输出: {result}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

输入: What a [MASK] day!   输出: What a lovely day!
输入: What a [MASK] film!  输出: What a wonderful film!
输入: What a [MASK] story! 输出: What a wonderful story!


In [6]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    pipeline
)
import torch

# NER标签集（CoNLL2003标准，B-前缀表示实体开头，I-表示中间）
LABEL_LIST = [
    "O",       # 非实体
    "B-PER",   # 人名-开头
    "I-PER",   # 人名-中间
    "B-LOC",   # 地名-开头
    "I-LOC",   # 地名-中间
    "B-ORG",   # 机构名-开头
    "I-ORG"    # 机构名-中间
]
MODEL_NAME = "bert-base-cased"  # 预训练基座，可替换为自定义微调模型

In [8]:
# 1. 加载模型与分词器
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label={i: label for i, label in enumerate(LABEL_LIST)},
    label2id={label: i for i, label in enumerate(LABEL_LIST)}
)
model.eval()

# 2. 输入文本（书中NER示例逻辑：含人名、地名的句子）
text = "Jack lives in New York and works at Google."

# 3. 分词（保留词元与原文本的映射关系）
inputs = tokenizer(
    text,
    return_tensors="pt",
    return_offsets_mapping=True  # 关键：获取词元在原文本的位置
)
offset_mapping = inputs.pop("offset_mapping")  # 移除模型不接收的参数

# 4. 前向传播
with torch.no_grad():
    outputs = model(** inputs)
logits = outputs.logits
predictions = torch.argmax(logits, dim=-1).squeeze().tolist()  # 取最大概率的标签

# 5. 结果解析（还原为原文本的实体）
entities = []
current_entity = {"text": "", "label": "", "start": -1, "end": -1}

for idx, (pred, offset) in enumerate(zip(predictions, offset_mapping[0])):

    # 👇 核心修复：使用 .item() 将 0维张量转换为普通 Python 整数
    token_id = inputs["input_ids"][0][idx].item()
    token_str = tokenizer.convert_ids_to_tokens(token_id)

    # 跳过特殊词元（[CLS]/[SEP]）和非实体（O）
    if token_str in ["[CLS]", "[SEP]"] or LABEL_LIST[pred] == "O":
        if current_entity["text"]:  # 结束当前实体
            entities.append(current_entity)
            current_entity = {"text": "", "label": "", "start": -1, "end": -1}
        continue

    label = LABEL_LIST[pred]
    start, end = offset
    token_text = text[start:end]

    # 处理实体开头（B-前缀）
    if label.startswith("B-"):
        if current_entity["text"]:  # 结束上一个实体
            entities.append(current_entity)
        current_entity = {
            "text": token_text,
            "label": label.split("-")[1],
            "start": start.item(), # offset 里的数字也可能是 tensor，稳妥起见加上 .item()
            "end": end.item()
        }
    # 处理实体中间（I-前缀）
    elif label.startswith("I-") and current_entity["label"] == label.split("-")[1]:
        # 拼接子词
        if start == current_entity["end"]:
            current_entity["text"] += token_text
        else:
            # 如果中间有空格，可以酌情加空格。但 BERT 使用 WordPiece，通常 ## 开头代表子词
            if token_str.startswith("##"):
                current_entity["text"] += token_text
            else:
                current_entity["text"] += " " + token_text
        current_entity["end"] = end.item()

# 补充最后一个实体
if current_entity["text"]:
    entities.append(current_entity)

# 打印手动推理结果
print("手动拆解NER结果：")
for ent in entities:
    print(f"实体类型: {ent['label']:5} 实体文本: {ent['text']:15} 位置: [{ent['start']}, {ent['end']}]")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

手动拆解NER结果：
实体类型: ORG   实体文本: New             位置: [14, 17]
实体类型: ORG   实体文本: York            位置: [18, 22]
实体类型: PER   实体文本: at Google       位置: [33, 42]


In [13]:
from datasets import load_dataset

# 使用 HF 官方核心维护者提供的纯静态 Parquet 镜像版本，完美绕过安全限制！
dataset = load_dataset("lhoestq/conll2003")

# 提取第一条训练数据
example = dataset["train"][0]

# 打印出来看看
print("成功提取第一条数据：")
print(example)

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

成功提取第一条数据：
{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [14]:
label2id = {
    "O": 0,
    "B-PER": 1, "I-PER": 2,
    "B-ORG": 3, "I-ORG": 4,
    "B-LOC": 5, "I-LOC": 6,
    "B-MISC": 7, "I-MISC": 8
}

id2label = {index: label for label, index in label2id.items()}

In [16]:
from datasets import load_dataset

# ⚠️ 注意这里！一定要加上 "lhoestq/" 前缀！
dataset = load_dataset("lhoestq/conll2003")

# 提取并打印第一条数据
example = dataset["train"][0]
print(example)

{'id': '0', 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [17]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [18]:
# 将单个词元拆分成子词元
token_ids = tokenizer(example["tokens"], is_split_into_words=True)["input_ids"]
sub_tokens = tokenizer.convert_ids_to_tokens(token_ids)
sub_tokens

['[CLS]',
 'EU',
 'rejects',
 'German',
 'call',
 'to',
 'boycott',
 'British',
 'la',
 '##mb',
 '.',
 '[SEP]']

In [19]:
def align_labels(examples):
    token_ids = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )
    labels = examples["ner_tags"]

    updated_labels = []
    for index, label in enumerate(labels):
        # 将词元映射到它们各自的单词
        word_ids = token_ids.word_ids(batch_index=index)
        previous_word_idx = None
        label_ids = []

In [23]:
# 假设 tokenizer 已经加载好
def tokenize_and_align_labels(examples):
    # 1. 对整批句子进行分词 (注意 is_split_into_words=True，因为原数据是按词切好的列表)
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    # 遍历批次中的每一条数据
    for i, original_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # 获取子词到原词的映射
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # 规则 1: 将特殊词元 ([CLS], [SEP]) 标记为 -100，让 PyTorch 计算 Loss 时忽略它们
            if word_idx is None:
                label_ids.append(-100)

            # 规则 2: 新单词的第一个子词，直接使用原始标签
            elif word_idx != previous_word_idx:
                label_ids.append(original_labels[word_idx])

            # 规则 3: 同一个单词的后续子词（比如被切开的词尾）
            else:
                current_label = original_labels[word_idx]
                # 在 CoNLL-2003 中，奇数通常是 B-XXX (如 1=B-PER)，将其加 1 变成偶数 I-XXX (如 2=I-PER)
                if current_label % 2 == 1:
                    current_label += 1
                label_ids.append(current_label)

            previous_word_idx = word_idx

        labels.append(label_ids)

    # 将对齐后的标签加回字典中
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [26]:
from transformers import AutoTokenizer

# 1. 加载专门处理分词的 Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# 2. 定义我们在上一步写好的“标配版”对齐规则
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    for i, original_labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # 特殊字符忽略
            elif word_idx != previous_word_idx:
                label_ids.append(original_labels[word_idx]) # 新词头保留原标签
            else:
                current_label = original_labels[word_idx]
                # B-标签(奇数)转为I-标签(偶数)
                if current_label % 2 == 1:
                    current_label += 1
                label_ids.append(current_label)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 3. 批量处理数据集！这步会生成你缺失的 tokenized_datasets
print("⏳ 正在批量处理数据集，请稍候...")
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# 4. 提取第一条数据进行对比展示
original_example = dataset["train"][0]
tokenized_example = tokenized_datasets["train"][0]

print("\n=== 🌟 原始数据 (按完整单词) ===")
for token, label in zip(original_example["tokens"], original_example["ner_tags"]):
    print(f"{token:15} : {label}")

print("\n" + "="*30 + "\n")

subwords = tokenizer.convert_ids_to_tokens(tokenized_example["input_ids"])
updated_labels = tokenized_example["labels"]

print("=== ✂️ 对齐后的数据 (按切分后的子词) ===")
for subword, label in zip(subwords, updated_labels):
    print(f"{subword:15} : {label}")

⏳ 正在批量处理数据集，请稍候...


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]


=== 🌟 原始数据 (按完整单词) ===
EU              : 3
rejects         : 0
German          : 7
call            : 0
to              : 0
boycott         : 0
British         : 7
lamb            : 0
.               : 0


=== ✂️ 对齐后的数据 (按切分后的子词) ===
[CLS]           : -100
EU              : 3
rejects         : 0
German          : 7
call            : 0
to              : 0
boycott         : 0
British         : 7
la              : 0
##mb            : 0
.               : 0
[SEP]           : -100


In [30]:
!pip install evaluate seqeval
import evaluate
import numpy as np

# 1. 动态获取 CoNLL-2003 数据集的真实文本标签列表
# 这样它就能把数字（0, 1, 2...）翻译成 ('O', 'B-PER', 'I-PER'...)
# 1. 直接提供 CoNLL-2003 数据集固定的 9 种真实文本标签
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

# 2. 加载序列评估器
seqeval = evaluate.load("seqeval")

# 3. 定义评估函数
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # 剔除掉那些标为 -100 的特殊词元，并把数字转换回文本标签
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # 调用 seqeval 计算 F1、精确度、召回率等指标
    results = seqeval.compute(
        predictions=true_predictions, references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

print("✅ 评估函数 compute_metrics 已准备就绪！")

✅ 评估函数 compute_metrics 已准备就绪！


In [ ]:
from transformers import DataCollatorForTokenClassification

# 词元分类数据整理器
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [31]:
from transformers import TrainingArguments, Trainer

# 用于参数调优的训练参数
training_args = TrainingArguments(
    output_dir="ner_bert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to="none"
)

# 初始化训练器
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [33]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

# 1. 专门用于 NER 任务的数据整理器，它会自动把标签也 pad 成 -100
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 2. 训练参数设置
training_args = TrainingArguments(
    output_dir="ner_bert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",     # 💡 顺便加上这行，让它每跑完一轮就评估一次 F1 分数！
    logging_strategy="steps",  # 开启文本进度播报
    logging_steps=50,          # 每 50 步打印一次 Loss
    report_to="none"
)

# 3. 初始化训练器
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"], # 👈 修复：使用对齐好的数字集
    eval_dataset=tokenized_datasets["test"],   # 👈 修复：使用对齐好的数字集
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

# 【防崩溃补丁】移除会导致报错的 Notebook 进度条组件
trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if "Notebook" not in cb.__class__.__name__
]

# 4. 终于可以开始训练啦！
print("🚀 开始微调 NER 模型...")
trainer.train()

🚀 开始微调 NER 模型...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2634, training_loss=0.07720178607837123, metrics={'train_runtime': 525.5325, 'train_samples_per_second': 80.153, 'train_steps_per_second': 5.012, 'total_flos': 1050534559887048.0, 'train_loss': 0.07720178607837123, 'epoch': 3.0})